# Notebook 2 — Machine Learning & LLM
**Proyecto:** Predicción de Incumplimiento en Créditos de Consumo

**Modelos:** Logistic Regression (baseline) vs LightGBM

**LLM:** Explicación automática de riesgo crediticio en lenguaje natural via Groq

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, log_loss
from scipy.stats import ks_2samp
from lightgbm import LGBMClassifier
import shap

RANDOM_STATE = 42
TARGET       = 'incumplimiento'
FEATURES = [
    'n_atr_1d_6', 'cl_9_1.0', 'cl_12_1.0', 'cl_18_2.0', 'cl_9_3.0',
    'ind_vjrc_36_1', 'ind_per_x9_1.0', 'prc_u_tc_cns', 'tas_u_tc',
    'c_tc_mn_24', 'c_tc_mn_24_sld', 'c_pcns', 'vr_s_t_36', 'mx_ltc_36', 'ipc_t6'
]

X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Train: (33580, 15) | Test: (8396, 15)


## 1. Modelo Baseline — Regresión Logística

In [2]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE,
        class_weight='balanced'
    ))
])
pipe_lr.fit(X_train, y_train)
prob_lr = pipe_lr.predict_proba(X_test)[:,1]

auc_lr   = roc_auc_score(y_test, prob_lr)
gini_lr  = 2 * auc_lr - 1
ks_lr    = ks_2samp(prob_lr[y_test==1], prob_lr[y_test==0]).statistic
brier_lr = brier_score_loss(y_test, prob_lr)
ll_lr    = log_loss(y_test, prob_lr)

print('=== Regresión Logística (Baseline) ===')
print(f'AUC        : {auc_lr:.4f}')
print(f'Gini       : {gini_lr:.4f}')
print(f'KS         : {ks_lr:.4f}')
print(f'Brier Score: {brier_lr:.4f}')
print(f'Log Loss   : {ll_lr:.4f}')

=== Regresión Logística (Baseline) ===
AUC        : 0.6844
Gini       : 0.3688
KS         : 0.2806
Brier Score: 0.2224
Log Loss   : 0.6357


## 2. Modelo Principal — LightGBM

In [3]:
lgbm = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    verbose=-1
)
lgbm.fit(X_train, y_train)
prob_lgbm = lgbm.predict_proba(X_test)[:,1]

auc_lgbm   = roc_auc_score(y_test, prob_lgbm)
gini_lgbm  = 2 * auc_lgbm - 1
ks_lgbm    = ks_2samp(prob_lgbm[y_test==1], prob_lgbm[y_test==0]).statistic
brier_lgbm = brier_score_loss(y_test, prob_lgbm)
ll_lgbm    = log_loss(y_test, prob_lgbm)

print('=== LightGBM ===')
print(f'AUC        : {auc_lgbm:.4f}')
print(f'Gini       : {gini_lgbm:.4f}')
print(f'KS         : {ks_lgbm:.4f}')
print(f'Brier Score: {brier_lgbm:.4f}')
print(f'Log Loss   : {ll_lgbm:.4f}')

=== LightGBM ===
AUC        : 0.7087
Gini       : 0.4174
KS         : 0.3029
Brier Score: 0.2115
Log Loss   : 0.6109


## 3. Comparación de modelos

In [4]:
resultados = pd.DataFrame({
    'Modelo'     : ['Logistic Regression', 'LightGBM'],
    'AUC'        : [auc_lr,   auc_lgbm],
    'Gini'       : [gini_lr,  gini_lgbm],
    'KS'         : [ks_lr,    ks_lgbm],
    'Brier Score': [brier_lr, brier_lgbm],
    'Log Loss'   : [ll_lr,    ll_lgbm],
})
display(resultados.round(4))

fpr_lr,   tpr_lr,   _ = roc_curve(y_test, prob_lr)
fpr_lgbm, tpr_lgbm, _ = roc_curve(y_test, prob_lgbm)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_lr, y=tpr_lr, mode='lines',
    name=f'Logistic Regression (AUC={auc_lr:.4f})',
    line=dict(color='#378ADD', width=2)))
fig.add_trace(go.Scatter(x=fpr_lgbm, y=tpr_lgbm, mode='lines',
    name=f'LightGBM (AUC={auc_lgbm:.4f})',
    line=dict(color='#1D9E75', width=2.5)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
    name='Aleatorio', line=dict(color='#888780', dash='dash', width=1.5)))
fig.update_layout(
    title='Curvas ROC — Comparación de modelos',
    xaxis_title='1 − Especificidad (FPR)',
    yaxis_title='Sensibilidad (TPR)',
    height=480, template='plotly_white'
)
fig.show()

,Modelo,AUC,Gini,KS,Brier Score,Log Loss
0,Logistic Regression,0.6844,0.3688,0.2806,0.2224,0.6357
1,LightGBM,0.7087,0.4174,0.3029,0.2115,0.6109


## 4. Análisis SHAP — Explicabilidad del modelo

In [5]:
explainer   = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(X_test)

sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_importance = pd.DataFrame({
    'feature'   : FEATURES,
    'shap_mean' : np.abs(sv).mean(axis=0)
}).sort_values('shap_mean', ascending=True)

fig = go.Figure(go.Bar(
    x=shap_importance['shap_mean'],
    y=shap_importance['feature'],
    orientation='h',
    marker_color='#D85A30'
))
fig.update_layout(
    title='Importancia de variables — SHAP (LightGBM)<br><sup>Mayor valor = mayor impacto en la predicción</sup>',
    xaxis_title='|SHAP value| promedio',
    height=500, template='plotly_white'
)
fig.show()

print('\nTop 5 variables más importantes:')
print(shap_importance.sort_values('shap_mean', ascending=False).head(5).to_string(index=False))


Top 5 variables más importantes:
     feature  shap_mean
   mx_ltc_36   0.350090
   vr_s_t_36   0.182890
    tas_u_tc   0.173088
  n_atr_1d_6   0.168107
prc_u_tc_cns   0.127190


## 5. Guardar artefactos

In [6]:
with open('../artifacts/logistic_regression.pkl', 'wb') as f:
    pickle.dump(pipe_lr, f)

with open('../artifacts/lightgbm_model.pkl', 'wb') as f:
    pickle.dump(lgbm, f)

resultados.to_csv('../artifacts/model_metrics.csv', index=False)
shap_importance.sort_values('shap_mean', ascending=False).to_csv(
    '../artifacts/shap_importance.csv', index=False
)

print('✅ Artefactos guardados en ../artifacts/')

✅ Artefactos guardados en ../artifacts/


## 6. LLM — Explicación de riesgo en lenguaje natural

Usamos el módulo `credit_risk.py` que implementa POO y principios SOLID para generar explicaciones automáticas del riesgo de cada cliente.

In [7]:
import sys
sys.path.append('..')
from credit_risk import LightGBMPredictor, CreditRiskExplainer, CreditRiskPipeline

# Crear .env si no existe
import os
if not os.path.exists('../.env'):
    with open('../.env', 'w', encoding='utf-8') as f:
        f.write('GROQ_API_KEY=TU_API_KEY_AQUI\n')
    print('⚠️  Crea el archivo .env con tu GROQ_API_KEY')

predictor = LightGBMPredictor('../artifacts/lightgbm_model.pkl')
explainer = CreditRiskExplainer(sv, FEATURES)
pipeline  = CreditRiskPipeline(predictor, explainer)

print('✅ Pipeline inicializado')

✅ Pipeline inicializado


In [8]:
# Evaluar cliente aleatorio
resultado = pipeline.evaluar_aleatorio(X_test, y_test)

print(f"Cliente idx : {resultado['cliente_idx']}")
print(f"Probabilidad: {resultado['probabilidad']:.1%}")
print(f"Nivel riesgo: {resultado['nivel_riesgo']}")
print(f"Real        : {'Incumplió' if resultado['real']==1 else 'Cumplió'}")
print(f"\n--- Explicación del LLM ---\n")
print(resultado['explicacion'])

Cliente idx : 447
Probabilidad: 63.1%
Nivel riesgo: ALTO
Real        : Incumplió

--- Explicación del LLM ---

**Resumen del riesgo del cliente**

El modelo LightGBM predijo una probabilidad de incumplimiento del cliente de 63.1%, lo que sitúa al cliente en un nivel de riesgo alto. Esto significa que existen condiciones que sugieren un mayor riesgo de que el cliente no cumpla con sus deudas.

**Factores que aumentan el riesgo**

El análisis de variables sugiere que las siguientes condiciones aumentan el riesgo del cliente:

* El cliente ha utilizado un alto porcentaje de la capacidad de crédito en su tarjeta de crédito en el último mes y en el máximo de los últimos 36 meses, lo que puede indicar un uso inadecuado de la tarjeta de crédito.
* El hecho de que el cliente tuvo créditos vencidos, refinanciados o castigados en los últimos 36 meses implica problemas anteriores con el pago de deudas.

**Factores que reducen el riesgo**

Por otro lado, los siguientes factores sugieren que el cli